# 01 — Data Understanding & Quality Assessment

This notebook inspects the working Excel workbook. Upload the file from your computer when prompted; raw data is not stored in this repository.

**Provenance caution:** the inspected workbook contains 10,194 rows, including 200 Canadian records and dates through 2026-12-30. Its exact source/version is unverified and differs from the commonly circulated 9,994-row US-only version.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io

# Upload the Excel workbook
uploaded = files.upload()
filename = next(iter(uploaded))

# Load data and preserve an untouched copy
df = pd.read_excel(io.BytesIO(uploaded[filename]))
df_raw = df.copy()
print('Uploaded file:', filename)
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
# Inspect schema, missing values, duplicates, and summary statistics
df.info()
display(df.isnull().sum())
print('Exact duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)


In [ ]:
# Validate dates, country/year distribution, and entity counts
print('Date range:', df['Order Date'].min(), 'to', df['Order Date'].max())
display(df['Country/Region'].value_counts())
display(df['Order Date'].dt.year.value_counts().sort_index())
print('Unique orders:', df['Order ID'].nunique())
print('Unique customers:', df['Customer ID'].nunique())
print('Unique products:', df['Product ID'].nunique())

# Reproducible future-date check using the project inspection date
cutoff_date = pd.Timestamp('2026-11-06')
future_rows = df[df['Order Date'] > cutoff_date]
print('Future-dated rows:', len(future_rows))
print('Future-dated percentage:', round(len(future_rows) / len(df) * 100, 2), '%')
print('Ship date before order date:', (df['Ship Date'] < df['Order Date']).sum())
print('Non-positive sales:', (df['Sales'] <= 0).sum())
print('Non-positive quantity:', (df['Quantity'] <= 0).sum())
print('Discount outside 0–1:', ((df['Discount'] < 0) | (df['Discount'] > 1)).sum())


## Data preparation notes

- Preserve the original DataFrame in `df_raw`.
- Use `df_clean` as a separate working copy.
- Keep Postal Code as text: values include Canadian alphanumeric postal codes and ZIP codes with possible leading zeros.
- Do not remove future-dated rows until source and analysis cutoff are decided.


In [ ]:
# Create a working copy and standardize Postal Code as text
df_clean = df_raw.copy()
df_clean['Postal Code'] = df_clean['Postal Code'].astype('string').str.strip()

# Trim leading/trailing whitespace from text columns
text_cols = df_clean.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

print('Cleaned working shape:', df_clean.shape)
print('Missing values:', df_clean.isnull().sum().sum())
print('Duplicate rows:', df_clean.duplicated().sum())
print('Duplicate Row IDs:', df_clean['Row ID'].duplicated().sum())
print('Postal Code dtype:', df_clean['Postal Code'].dtype)


## Findings from the inspected workbook

- Shape: 10,194 rows × 21 columns.
- Country distribution: United States 9,994; Canada 200.
- Date range: 2023-01-03 to 2026-12-30.
- Missing values and exact duplicate rows: 0.
- Future-dated rows relative to 2026-11-06: 828 (8.12%).
- No ship dates before order dates, non-positive sales/quantity, or discounts outside 0–1 were found.

These observations apply only to the inspected local workbook. Re-run the notebook after uploading the file to reproduce them. Provenance remains unverified.
